In [9]:
import pandas as pd
import xgboost as xgb
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report, roc_auc_score, precision_score, recall_score, f1_score
import orjson
import neurokit2 as nk
def get_json(path):
    with open(path, 'r') as f:
        return orjson.loads(f.read())

In [10]:
# --- 1. Efficiently Load the JSON Data (Assuming JSON Lines format) ---
filepath = '../results/combined_prna_outputs_modified.json' # Replace with your file path

raw_data = get_json(filepath)
# --- Step 2: Flatten the Nested Structure ---

In [3]:
raw_data[0]

{'exam_id': '763256',
 'chagas': False,
 'age': '21',
 'is_male': 'False',
 'nn_predicted_age': '32.933292',
 '1dAVb': 'False',
 'RBBB': 'False',
 'LBBB': 'False',
 'SB': 'False',
 'ST': 'False',
 'AF': 'False',
 'patient_id': '1217369',
 'death': 'False',
 'timey': '2.961641',
 'normal_ecg': 'False',
 'trace_file': 'exams_part1.hdf5',
 'primary_id': 'code15_763256',
 'source': 'code15',
 'snomed_vals': {'10370003': {'present': False, 'probability': 8.375992e-05},
  '111975006': {'present': False, 'probability': 0.0010521555},
  '164889003': {'present': False,
   'probability': 0.00013104455,
   'actual_present': False},
  '164890007': {'present': False, 'probability': 8.582029e-06},
  '164909002': {'present': False,
   'probability': 8.851866e-06,
   'actual_present': False},
  '164917005': {'present': False, 'probability': 0.0008262891},
  '164934002': {'present': False, 'probability': 0.0024428766},
  '164947007': {'present': False, 'probability': 1.0487474e-06},
  '17338001': {'pre

In [4]:
import os
import scipy.io
from typing import Dict, Any, Optional

def load_ecg_data(
    exam_id: str,
    search_directory: str,
    source: Optional[str] = None,
    verbose: bool = False
) -> Optional[Dict[str, Any]]:
    """
    Recursively searches for an ECG data file (.mat) corresponding to a given
    exam_id and returns the loaded data. An optional source parameter can be
    provided to specify a subdirectory within the search_directory.

    Args:
        exam_id: The identifier of the exam.
        search_directory: The root directory to start the search from.
        source: An optional subdirectory within search_directory to limit the search.

    Returns:
        Tuple of .mat file data, and the .hea filename if found, otherwise None.
    """
    mat_filename = f"{exam_id}.mat"
    hea_filename = f"{exam_id}.hea"

    # Construct the final search path
    effective_search_path = search_directory
    if source:
        effective_search_path = os.path.join(search_directory, source)

    # Check if the effective search path exists
    if not os.path.isdir(effective_search_path):
        print(f"Error: Search path does not exist: {effective_search_path}")
        return None

    if verbose:
        print(f"Searching for {exam_id} in '{effective_search_path}'...")
    for dirpath, _, filenames in os.walk(effective_search_path):
        if mat_filename in filenames and hea_filename in filenames:
            mat_filepath = os.path.join(dirpath, mat_filename)
            try:
                if verbose:
                    print(f"Found {mat_filename} at: {mat_filepath}")
                # Load the .mat file
                mat_data = scipy.io.loadmat(mat_filepath)
                return mat_data['val'], os.path.join(dirpath, hea_filename)
            except Exception as e:
                print(f"Error loading {mat_filepath}: {e}")
                return None
    
    print(f"No .mat and .hea files found for exam_id: {exam_id} in the specified path.")
    return None

In [11]:
import sys
sys.path.append("../") # Or os.path.dirname(script_dir) if helper_code is up one level
import helper_code
processed_records = []
error_records = []
for record in raw_data:
    # Start a new dictionary for the flattened record
    # Include any other top-level data you need, like an ID or the target variable
    flat_record = {
        'exam_id': record['exam_id'],
        'chagas': record['chagas']
    }
    # Iterate through the list of SNOMED code dictionaries
    snomed_vals = None
    if "snowmed_vals"  in record:
        snomed_vals = record['snowmed_vals']
    else:
        snomed_vals = record['snomed_vals']

    for snomed_item in snomed_vals:
        # The key is the SNOMED code (e.g., "44054006")
        snomed_code = snomed_item
        # The value is the "present" boolean|
        is_present = snomed_vals[snomed_code]['present']

        # Add the SNOMED code as a feature, converting boolean to 0 or 1
        flat_record[snomed_code] = int(is_present)
        source = record['source']

        # signal, header_path = load_ecg_data(record['exam_id'], "../training_data", source)
        # frequency = int(helper_code.get_sampling_frequency(helper_code.load_header(header_path)))
        # extra_features = process_12_lead_ecg(signal, frequency)
        # flat_record.update(extra_features)

    processed_records.append(flat_record)

In [11]:
processed_records[-1]

{'exam_id': '10484_hr',
 'chagas': False,
 '10370003': 0,
 '17338001': 0,
 '39732003': 0,
 '47665007': 0,
 '59118001': 0,
 '59931005': 0,
 '63593006': 0,
 '111975006': 0,
 '164889003': 0,
 '164890007': 0,
 '164909002': 0,
 '164917005': 0,
 '164934002': 0,
 '164947007': 0,
 '251146004': 0,
 '270492004': 0,
 '284470004': 0,
 '426177001': 1,
 '426627000': 0,
 '426783006': 1,
 '427084000': 0,
 '427172004': 0,
 '427393009': 0,
 '445118002': 0,
 '698252002': 0,
 '713426002': 0,
 '713427006': 0}

In [4]:
processed_records[0]

{'exam_id': '763256',
 'chagas': False,
 '10370003': 0,
 '111975006': 0,
 '164889003': 0,
 '164890007': 0,
 '164909002': 0,
 '164917005': 0,
 '164934002': 0,
 '164947007': 0,
 '17338001': 0,
 '251146004': 0,
 '270492004': 0,
 '284470004': 0,
 '39732003': 0,
 '426177001': 0,
 '426627000': 0,
 '426783006': 1,
 '427084000': 0,
 '427172004': 0,
 '427393009': 0,
 '445118002': 0,
 '47665007': 0,
 '59118001': 0,
 '59931005': 0,
 '63593006': 0,
 '698252002': 0,
 '713426002': 0,
 '713427006': 0}

# Get record paths

In [1]:
import utils
import helper_code
records = utils.prepare_stratification(helper_code.find_records_abs("../training_data"))

Starting parallel processing for 366185 records with max_workers=8...


Extracting labels and sources:   0%|          | 0/366185 [00:00<?, ?it/s]

In [ ]:
import feature_eda_pipeline_GPT as featurize
records_paths = [record['record'] for record in records]
df = featurize.run_feature_extraction_for_records_absolute(records_paths, "feature_importance_ranking.csv", 20, -1)

Will only extract these top 20 features: {'V4_var', 'V5_std', 'V5_range', 'wavelet_energy_d1', 'I_std', 'V6_var', 'wavelet_std_a4', 'V6_range', 'I_range', 'V1_var', 'V6_std', 'aVL_range', 'V4_range', 'V1_std', 'I_var', 'V4_std', 'aVL_var', 'V1_range', 'V5_var', 'V3_range'}
Starting feature extraction for 366185 records...










/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(














/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty ve































/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning 

/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
























/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_fixpeaks.py:307: RuntimeWarning: invalid value encountered in divide
  mrrs /= th2
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_fixpeaks.py:307: RuntimeWarning: invalid value encountered in divide
  mrrs /= th2
/juice2/scr2/k

...

Global feature extraction failed for /juice2/scr2/kelvinkn/other_work/edwards/physionet2025/training_data/code15/code15_outputs/code15_half1/exams_part3/1346982 channel 0: The data length is too small to be segmented.
Global feature extraction failed for /juice2/scr2/kelvinkn/other_work/edwards/physionet2025/training_data/code15/code15_outputs/code15_half1/exams_part3/1346982 channel 1: The data length is too small to be segmented.
Global feature extraction failed for /juice2/scr2/kelvinkn/other_work/edwards/physionet2025/training_data/code15/code15_outputs/code15_half1/exams_part3/1346982 channel 2: The data length is too small to be segmented.



/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(





/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(




/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/juice2/scr2/kelvin

Global feature extraction failed for /juice2/scr2/kelvinkn/other_work/edwards/physionet2025/training_data/code15/code15_outputs/code15_half1/exams_part3/1366263 channel 3: integer division or modulo by zero




/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/neurokit2/signal/signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/juice2/scr2/kelvinkn/conda_envs/physionet/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/juice2/scr2/kelv

In [7]:
df[0].iloc[0]

V1_std                                                        0.186298
V1_var                                                        0.034707
V1_range                                                         1.295
V3_range                                                         2.425
V4_std                                                        0.264838
V4_var                                                        0.070139
V4_range                                                         2.388
V5_std                                                        0.296805
V5_var                                                        0.088093
V5_range                                                         2.557
V6_std                                                         0.26617
V6_var                                                        0.070847
V6_range                                                         2.093
I_std                                                         0.222797
I_var 

In [19]:
processed_records[0]['exam_id']

'763256'

In [20]:
# Convert the list of dictionaries to a DataFrame
processed_df = pd.DataFrame(processed_records)

# Extract the features DataFrame from the tuple returned by the function
features_df = df[0]

# --- Merge the DataFrames ---
# Ensure the 'exam_id' columns are of the same type (string) for a clean merge
features_df['exam_id'] = features_df['exam_id'].astype(str)
processed_df['exam_id'] = processed_df['exam_id'].astype(str)

# Perform a left merge to add SNOMED features to the extracted ECG features
# This keeps all records from `features_df` and adds matching data from `processed_df`
merged_df = pd.merge(features_df, processed_df, on='exam_id', how="inner")

# Display the first few rows of the merged DataFrame and its shape
print("Shape of the merged DataFrame:", merged_df.shape)
merged_df.head()

Shape of the merged DataFrame: (99, 50)


,V1_std,V1_var,V1_range,V3_range,V4_std,V4_var,V4_range,V5_std,V5_var,V5_range,...,427172004,427393009,445118002,47665007,59118001,59931005,63593006,698252002,713426002,713427006
0,0.186298,0.034707,1.295,2.425,0.264838,0.070139,2.388,0.296805,0.088093,2.557,...,0,0,0,0,0,0,0,0,0,0
1,0.839422,0.704629,4.369,8.624,2.387876,5.701951,11.815,1.909409,3.645841,11.741,...,0,0,0,0,0,0,0,0,0,0
2,0.661116,0.437074,4.228,3.071,0.681979,0.465096,4.310,0.924687,0.855046,5.125,...,0,0,0,0,0,0,0,0,0,0
3,0.276296,0.076340,2.161,1.866,0.328655,0.108014,2.319,0.336751,0.113402,2.359,...,0,0,0,0,0,0,0,0,0,0
4,0.166414,0.027694,1.432,1.539,0.319830,0.102291,2.447,0.343928,0.118286,3.689,...,0,0,0,0,1,0,0,0,0,0


In [21]:
merged_df.iloc[0]

V1_std                                                        0.186298
V1_var                                                        0.034707
V1_range                                                         1.295
V3_range                                                         2.425
V4_std                                                        0.264838
V4_var                                                        0.070139
V4_range                                                         2.388
V5_std                                                        0.296805
V5_var                                                        0.088093
V5_range                                                         2.557
V6_std                                                         0.26617
V6_var                                                        0.070847
V6_range                                                         2.093
I_std                                                         0.222797
I_var 